<a href="https://colab.research.google.com/github/Felipe-data2026/statsmodels/blob/main/sentiment_analysis_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Configuração inicial

Para não interromper a aula com instalações e imports no meio dos blocos, **tudo que será necessário** (bibliotecas do Bloco 1, 2 e 3) é instalado e importado de uma vez, nesta única célula. Essa célula deve ser executada antes de seguir — pode levar um ou dois minutos por causa do modelo do spaCy.

In [1]:
!pip install -q pandas scikit-learn matplotlib nltk textblob spacy groq
!python -m spacy download pt_core_news_sm -q
!git clone -q https://github.com/rafjaa/LeIA.git
!git clone -q https://github.com/guilhermeonrails/LeIA-MBA.git

import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# LeIA padrão e LeIA-MBA são dois arquivos "leia.py" diferentes (mesmo nome de módulo).
# Usamos importlib com caminho explícito para carregar os dois sem que um sobrescreva o outro.
def carregar_leia(caminho_modulo, nome_modulo):
    spec = importlib.util.spec_from_file_location(nome_modulo, caminho_modulo)
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    return modulo

SentimentIntensityAnalyzerPT = carregar_leia("LeIA/leia.py", "leia_padrao").SentimentIntensityAnalyzer
SentimentIntensityAnalyzerMBA = carregar_leia("LeIA-MBA/leia.py", "leia_mba").SentimentIntensityAnalyzer
from textblob import TextBlob
import spacy
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 76.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Recapitulando...

A função `classificar_sentimento` (léxico manual) da aula passada é repetida aqui. Ela serve de referência para a comparação final no Bloco 3.

In [2]:
def classificar_sentimento(frase):
    # Léxico simples de palavras em português
    palavras_positivas = [
        "bom", "boa", "ótimo", "excelente", "maravilhoso", "gostei", "gosto",
        "feliz", "amor", "adoro", "sensacional", "perfeito", "parabéns", "sucesso"
    ]
    palavras_negativas = [
        "ruim", "péssimo", "horrível", "odiei", "odeio", "triste", "pena",
        "chato", "difícil", "problema", "erro", "fracasso", "mal", "pior"
    ]

    frase_limpa = frase.lower()
    for pontuacao in [".", ",", "!", "?", ";", ":"]:
        frase_limpa = frase_limpa.replace(pontuacao, "")
    palavras = frase_limpa.split()

    positivos = 0
    negativos = 0
    for palavra in palavras:
        if palavra in palavras_positivas:
            positivos += 1
        elif palavra in palavras_negativas:
            negativos += 1

    if positivos > negativos:
        return "Positivo"
    elif negativos > positivos:
        return "Negativo"
    else:
        return "Neutro"

In [3]:
classificar_sentimento("O atendimento foi excelente, adorei a experiência!")

'Positivo'

In [4]:
classificar_sentimento("O atendimento foi ruim")

'Negativo'

In [5]:
classificar_sentimento("O atendimento foi ok")

'Neutro'

In [6]:
# Frases-desafio da aula anterior, usadas nos 3 blocos para comparar os métodos
frases_desafio = [
    "Não gostei da cor do produto.",
    "Ah, que ótimo, quebrou no primeiro dia!",
    "Não tive problema nenhum",
    "Mal posso esperar para usar",
    "Jogo difícil de largar, é muito viciante!!!",
]

In [7]:
for frase in frases_desafio:
    print(f"{classificar_sentimento(frase)} >>> {frase}")

Positivo >>> Não gostei da cor do produto.
Positivo >>> Ah, que ótimo, quebrou no primeiro dia!
Negativo >>> Não tive problema nenhum
Negativo >>> Mal posso esperar para usar
Negativo >>> Jogo difícil de largar, é muito viciante!!!


### Parte 3 — Análise de Sentimentos

Na aula anterior (`sentimento.ipynb`) foi construído um classificador **baseado em léxico**: palavras positivas e negativas foram contadas e o sentimento foi decidido pelo saldo.

Depois, foi demonstrado que um modelo Transformer pronto que resolve melhor os casos difíceis.

Agora o conteúdo é aprofundado em **3 blocos**:

1. **Treino de um classificador próprio** com Machine Learning, a partir de exemplos escritos à mão (200 frases em dois `.csv`).
2. **Apresentação das bibliotecas e ferramentas gratuitas** mais populares do mercado para análise de sentimento.
3. **Uso de um LLM (Google Gemini)** para classificar sentimentos com IA generativa.

---
# Bloco 1 — Classificando com Machine Learning

Em vez de decidir manualmente quais palavras são "positivas" ou "negativas", um algoritmo **aprende isso** a partir de exemplos.

### A ideia

1. Foram criados dois arquivos `positivas.csv` e `negativas.csv`.
2. O texto é transformado em números com **Bag of Words** (a mesma ideia de "contar palavras" da aula passada, só que agora de forma vetorizada).
3. Um modelo de **Naive Bayes** é treinado para aprender a associação entre palavras e sentimento.
4. O modelo é testado com frases novas, que ele nunca viu.

### 1.1) Carregando as bases de treino (.csv)

Cada `.csv` tem uma única coluna `frase`. Os dois arquivos são carregados, a coluna `sentimento` é adicionada e tudo é reunido em um único DataFrame.

In [8]:
df_positivas = pd.read_csv("positivas.csv")
df_negativas = pd.read_csv("negativas.csv")

In [9]:
df_positivas.head()

,frase,sentimento
0,"Adorei a panela, super recomendo a todos!",muito positiva
1,"Chegou tudo certinho, o notebook funcionando l...",positiva
2,"Estou encantado com a panela, valeu cada centavo.",positiva
3,"Que produto incrível, o brinquedo funciona per...",muito positiva
4,"Ótimo custo-benefício, o produto superou o que...",muito positiva


In [10]:
df_negativas.head()

,frase,sentimento
0,O relógio não atendeu nem de longe minhas expe...,muito negativa
1,"O tablet é frágil, feio e muito difícil de usar.",negativa
2,"Que produto decepcionante, o teclado parou de ...",negativa
3,Já é a segunda vez que tenho problema com os f...,negativa
4,"Péssima experiência, não recomendo o brinquedo...",negativa


In [11]:
df_positivas.shape, df_negativas.shape

((1000, 2), (1000, 2))

In [12]:
df_positivas["sentimento"] = "positivo"
df_negativas["sentimento"] = "negativo"
df_positivas.columns, df_negativas.columns

(Index(['frase', 'sentimento'], dtype='object'),
 Index(['frase', 'sentimento'], dtype='object'))

In [13]:
df = pd.concat([df_positivas, df_negativas], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # embaralhar
df.head(10)

,frase,sentimento
0,"A smart tv é horrível, uso poucas vezes e já a...",negativo
1,"Adorei a smart tv, super recomendo a todos!",positivo
2,Fiquei extremamente decepcionado com a churras...,negativo
3,"Estou muito feliz com a smart tv, já indiquei ...",positivo
4,A churrasqueira elétrica é de baixíssima quali...,negativo
5,A embalagem chegou toda amassada e o mouse vei...,negativo
6,"Superou todas as expectativas, a smart tv é si...",positivo
7,O carregador parou de funcionar em poucos dias...,negativo
8,A impressora tem ótima qualidade e o preço val...,positivo
9,Os fones bluetooth parou de funcionar em pouco...,negativo


In [14]:
print(df["sentimento"].value_counts())

sentimento
negativo    1000
positivo    1000
Name: count, dtype: int64


In [15]:
df.shape

(2000, 2)

### 1.2) Separando treino e teste

Foram reservados 50% das frases para **teste** (o modelo não vê essas frases durante o treino), e o parâmetro `stratify` é usado para manter a proporção 50/50 de positivas e negativas nos dois conjuntos.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    df["frase"], df["sentimento"],
    test_size=0.5, random_state=42, stratify=df["sentimento"]
)

print(f"Treino: {len(X_train)} frases | Teste: {len(X_test)} frases")

Treino: 1000 frases | Teste: 1000 frases


### 1.3) Bag of Words: transformando texto em números

O `CountVectorizer` faz exatamente o que foi feito manualmente na aula passada (`frase.lower().split()`), só que para o vocabulário inteiro: cada frase vira um vetor com a contagem de cada palavra do vocabulário.

**Importante:** o vocabulário é aprendido **apenas com o treino** (`fit_transform`). No teste, apenas `transform` é usado, simulando frases novas no mundo real.

In [17]:
# Cria um objeto que vai transformar
# textos em números com base na contagem de palavras.
vectorizer = CountVectorizer()

In [18]:
# Aprende quais palavras existem nos textos de treino
# e transforma esses textos em uma matriz numérica.
X_train_vec = vectorizer.fit_transform(X_train)

In [19]:
# Transforma os textos de teste em números usando
# as mesmas palavras aprendidas no treino.
X_test_vec = vectorizer.transform(X_test)

In [20]:
print("Tamanho do vocabulário:", len(vectorizer.vocabulary_))

Tamanho do vocabulário: 243


### 1.4) Treinando o modelo (Naive Bayes)

`MultinomialNB` é o algoritmo clássico para classificação de texto: ele calcula, para cada palavra, a probabilidade dela aparecer em frases positivas vs. negativas — e usa isso para classificar frases novas.

In [21]:
modelo = MultinomialNB()
modelo.fit(X_train_vec, y_train)

MultinomialNB()

### 1.5) Avaliando no conjunto de teste

In [22]:
y_pred = modelo.predict(X_test_vec)
print("Acurácia:", accuracy_score(y_test, y_pred))

Acurácia: 1.0


In [23]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negativo       1.00      1.00      1.00       500
    positivo       1.00      1.00      1.00       500

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



### 1.6) Função de classificação com o modelo treinado

Tudo (vetorização + predição + probabilidade) é empacotado em uma função, no mesmo estilo da `classificar_sentimento` da aula anterior.

In [24]:
def classificar_com_ml(frase):
    vetor = vectorizer.transform([frase])
    predicao = modelo.predict(vetor)[0]
    probabilidade = modelo.predict_proba(vetor).max()
    return predicao, probabilidade

In [25]:
frase = "Não tive problema nenhum"
sentimento, confianca = classificar_com_ml(frase)
print(f"{sentimento} ({confianca:.0%} de confiança)")

negativo (99% de confiança)


In [26]:
frase = "Esse produto farmou aura aqui em casa. Não tanquei o quão bom ele é"
sentimento, confianca = classificar_com_ml(frase)
print(f"{sentimento} ({confianca:.0%} de confiança)")

positivo (92% de confiança)


In [27]:
frase = "Não gostei do cor do produto"
sentimento, confianca = classificar_com_ml(frase)
print(f"{sentimento} ({confianca:.0%} de confiança)")

negativo (80% de confiança)


### 1.7) Testando com frases novas (fora do treino)

As mesmas **frases-desafio** da aula anterior são usadas — aquelas que confundiram o método de léxico manual — para verificar se o modelo treinado se sai melhor.

In [28]:
print(f'-'*81)
print(f"{'ML (Naive Bayes)':22} | {'Confiança':10} | Frase")
print(f'-'*81)
for frase in frases_desafio:
    sentimento, confianca = classificar_com_ml(frase)
    print(f"- {sentimento:20} | {confianca:10.0%} | {frase}")
print(f'-'*81)

---------------------------------------------------------------------------------
ML (Naive Bayes)       | Confiança  | Frase
---------------------------------------------------------------------------------
- negativo             |        99% | Não gostei da cor do produto.
- positivo             |        99% | Ah, que ótimo, quebrou no primeiro dia!
- negativo             |        99% | Não tive problema nenhum
- negativo             |        94% | Mal posso esperar para usar
- negativo             |        92% | Jogo difícil de largar, é muito viciante!!!
---------------------------------------------------------------------------------


### O que aconteceu?

O modelo treinado acerta alguns casos que o léxico manual errava (como *"Ah, que ótimo, quebrou no primeiro dia!"*, agora corretamente negativo), porque ele aprendeu que palavras como *"quebrou"* pesam mais do que *"ótimo"* nos exemplos de treino.


### Desafio

- Novas frases (positivas e negativas) podem ser adicionadas aos `.csv`, com o modelo sendo retreinado em seguida. A acurácia melhora?

---
# Bloco 2 — Ferramentas e bibliotecas para análise de sentimento

Ferramentas **gratuitas e open-source**  usadas no mercado.

| Ferramenta | O que é | Funciona bem em PT-BR? |
|---|---|---|
| **NLTK (VADER)** | Toolkit de NLP mais tradicional, com o léxico VADER para sentimento | ❌ Léxico é em inglês |
| **LeIA** | Adaptação do VADER feita para português, por pesquisadores brasileiros | ✅ Sim |
| **TextBlob** | Lib simples e popular, API muito fácil de usar | ❌ Focada em inglês |
| **spaCy** | Framework industrial de NLP (tokenização, lematização, NER) | ⚠️ Ótimo em PT-BR, mas não tem sentimento pronto |

## 2.1) NLTK + VADER

O [NLTK](https://www.nltk.org/) é uma das bibliotecas de NLP mais antigas e usadas em cursos e pesquisa. Ele inclui o **VADER** (`SentimentIntensityAnalyzer`), um léxico muito popular para sentimento — mas construído para **inglês**.

In [29]:
sia = SentimentIntensityAnalyzer()

In [30]:
sia.polarity_scores("Eu gostei muito do atendimento, foi excelente!")

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

Nota-se que o `compound` (score geral) fica em **0.0** — o VADER simplesmente não reconhece nenhuma palavra em português no seu léxico. Isso é um ótimo exemplo de por que **ferramentas feitas para a nossa língua são necessárias**.

In [31]:
sia.polarity_scores("I loved the service, it was excellent!")

{'neg': 0.0, 'neu': 0.336, 'pos': 0.664, 'compound': 0.8356}

### Resultado da análise de sentimento (VADER)

```python
{'neg': 0.0, 'neu': 0.336, 'pos': 0.664, 'compound': 0.8356}
```

- **neg: 0.0** → Proporção de sentimento negativo (**0% negativo**).
- **neu: 0.336** → Proporção de conteúdo neutro (**33,6% neutro**).
- **pos: 0.664** → Proporção de sentimento positivo (**66,4% positivo**).
- **compound: 0.8356** → Pontuação geral do sentimento, variando de **-1** a **+1**.

### Interpretação do `compound`

| Faixa | Classificação |
|---------|--------------|
| -1 a -0.05 | Negativo |
| -0.05 a 0.05 | Neutro |
| 0.05 a 1 | Positivo |

### Conclusão

Como o valor de **compound = 0.8356**, o texto apresenta um sentimento **fortemente positivo**.

## 2.2) LeIA — Léxico para Inferência Adaptado (PT-BR)

O [LeIA](https://github.com/rafjaa/LeIA) é uma adaptação do VADER **feita para português**, criada por pesquisadores brasileiros. Não está disponível via `pip` — por isso o repositório do GitHub já foi clonado na configuração inicial. Aqui, apenas a importação é necessária.

In [34]:
leia.polarity_scores ("Eu amei o produto")

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

In [33]:
leia = SentimentIntensityAnalyzerPT()

In [35]:
leia.polarity_scores("Eu gostei muito do atendimento, foi excelente!")

{'neg': 0.0, 'neu': 0.41, 'pos': 0.59, 'compound': 0.8016}

In [36]:
leia.polarity_scores("Não gostei do atendimento")

{'neg': 0.314, 'neu': 0.286, 'pos': 0.4, 'compound': 0.1531}

Agora sim! O `compound` reflete o sentimento real da frase. O teste é feito a seguir nas frases-desafio:

In [37]:
print(f"{'LeIA (compound)':16} | Frase")
for frase in frases_desafio:
    score = leia.polarity_scores(frase)["compound"]
    print(f"{score:16.3f} | {frase}")

LeIA (compound)  | Frase
           0.153 | Não gostei da cor do produto.
          -0.151 | Ah, que ótimo, quebrou no primeiro dia!
          -0.599 | Não tive problema nenhum
          -0.718 | Mal posso esperar para usar
          -0.313 | Jogo difícil de largar, é muito viciante!!!


In [38]:
leia.polarity_scores("Eu amei o produto")
# 🫠

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

### 2.2.2) LeIA-MBA: estendendo o léxico para cobrir as lacunas encontradas

A seção anterior mostrou o LeIA **padrão** confessando sua própria limitação: `"Eu amei o produto"`
volta como "Neutro (sem palavras reconhecidas)" porque o léxico simplesmente não tem as palavras
"amei", "amo" ou "adorei" — só o substantivo "amor".

Para resolver isso, o léxico foi **estendido**: 25 palavras novas (a família "amar"/"detestar",
"péssimo", "incrível", "impecável", "frágil", "decepcionante", "baixíssimo" e variações de gênero)
e 3 correções de score que estavam incoerentes com o uso real (ex.: "sensacional" tinha score
levemente negativo no léxico original!).

Esse léxico estendido está publicado como **[LeIA-MBA](https://github.com/guilhermeonrails/LeIA-MBA)**,
licenciado em **MIT** (mesma licença do LeIA original) — livre para qualquer aluno usar, copiar ou
estender no futuro.

**Importante:** não é um léxico "completo" — nenhum léxico é. É uma extensão pontual e documentada,
não uma reescrita.

In [39]:
leia_mba = SentimentIntensityAnalyzerMBA()

In [40]:
frases_extras = [
    "Eu amei o produto",
    "O atendimento foi péssimo",
    "A caixa de som é sensacional",
    "Estou muito frustrado com o produto",
    "O produto é frágil e de baixíssima qualidade",
]

In [41]:
print(f"{'Frase':45} | {'LeIA padrão':12} | {'LeIA-MBA':12}")
print("-" * 75)
for frase in frases_desafio + frases_extras:
    compound_padrao = leia.polarity_scores(frase)["compound"]
    compound_mba = leia_mba.polarity_scores(frase)["compound"]
    print(f"{frase:45} | {compound_padrao:<12.3f} | {compound_mba:<12.3f}")

Frase                                         | LeIA padrão  | LeIA-MBA    
---------------------------------------------------------------------------
Não gostei da cor do produto.                 | 0.153        | 0.153       
Ah, que ótimo, quebrou no primeiro dia!       | -0.151       | -0.151      
Não tive problema nenhum                      | -0.599       | -0.599      
Mal posso esperar para usar                   | -0.718       | -0.718      
Jogo difícil de largar, é muito viciante!!!   | -0.313       | -0.313      
Eu amei o produto                             | 0.000        | 0.586       
O atendimento foi péssimo                     | 0.000        | -0.612      
A caixa de som é sensacional                  | -0.077       | 0.586       
Estou muito frustrado com o produto           | -0.026       | -0.459      
O produto é frágil e de baixíssima qualidade  | 0.000        | -0.727      


### 2.2.3) Classificando em 5 categorias

O objetivo é classificar toda frase em exatamente uma destas 5 faixas:

**Muito negativo,
Negativo, Neutro, Positivo, Muito positivo**

### Por que não usar só o `compound`?

Como visto na 2.2.1, uma frase com carga positiva e negativa **ao mesmo tempo** (ex.: "Bom, porém ruim") pode ter as duas se cancelando no `compound`, resultando em um valor perto de zero — mas isso não é a mesma coisa que uma frase realmente sem carga emocional.

### Regra que será usada

1. **`neu >= 0.9`** → a frase é quase inteiramente neutra (pouquíssima carga emocional) → **Neutro** direto.
2. Se o `compound` já é decisivo (`<= -0.55` ou `>= 0.55`), ele manda: **Muito negativo** / **Muito positivo**.
3. Se o `compound` cai na zona neutra (`-0.2` a `0.2`) **mas** `neg` e `pos` são ambos relevantes (`>= 0.15`) — ou seja, a frase tem carga positiva **e** negativa que se cancelaram no `compound` — usa-se o **saldo** `pos - neg` para decidir pra que lado ela pende de verdade, em vez de aceitar cegamente "Neutro".
4. Nos demais casos, o `compound` é cortado nas faixas intermediárias.


| Condição | Categoria |
|---|---|
| `neu >= 0.9` | Neutro |
| `compound <= -0.55` | Muito negativo |
| `-0.55 < compound <= -0.2` | Negativo |
| `-0.2 < compound < 0.2` e `neg`/`pos` ambos `>= 0.15` | Positivo/Negativo, conforme o saldo `pos - neg` |
| `-0.2 < compound < 0.2` (demais casos) | Neutro |
| `0.2 <= compound < 0.55` | Positivo |
| `compound >= 0.55` | Muito positivo |

In [42]:
def classificar_sentimento_5_niveis(frase):
    scores = leia_mba.polarity_scores(frase)
    neg, neu, pos, compound = scores["neg"], scores["neu"], scores["pos"], scores["compound"]

    # Frase quase inteiramente neutra (pouca carga emocional) -> Neutro direto.
    if neu >= 0.9:
        return "Neutro"

    if compound <= -0.55:
        return "Muito negativo"
    elif compound <= -0.2:
        return "Negativo"
    elif compound < 0.2:
        # Zona neutra do compound: se neg e pos são ambos relevantes ao mesmo
        # tempo, a frase tinha carga positiva E negativa que se cancelaram no
        # compound -> usa-se o saldo (pos - neg) pra decidir pra que lado ela
        # pende de verdade, em vez de aceitar cegamente "Neutro".
        if neg >= 0.15 and pos >= 0.15:
            saldo = pos - neg
            if saldo >= 0.1:
                return "Positivo"
            elif saldo <= -0.1:
                return "Negativo"
        return "Neutro"
    elif compound < 0.55:
        return "Positivo"
    else:
        return "Muito positivo"

In [43]:
frase = "Não gostei da cor do produto"
classificar_sentimento_5_niveis(frase)

'Neutro'

## 2.4) TextBlob

O [TextBlob](https://textblob.readthedocs.io/) é conhecido pela API extremamente simples: `TextBlob(texto).sentiment` já retorna polaridade e subjetividade. É muito popular no mercado americano, mas — assim como o VADER — foi feito para **inglês**.

In [44]:
TextBlob("Eu gostei muito do atendimento, foi excelente!").sentiment

Sentiment(polarity=0.0, subjectivity=0.0)

In [45]:
TextBlob("I loved the service, it was excellent!").sentiment

Sentiment(polarity=0.85, subjectivity=0.9)

Em português, a `polarity` fica em **0.0** (nenhum sinal). Em inglês, a ferramenta identifica corretamente uma frase bem positiva. Fica o aprendizado: **o suporte da ferramenta ao idioma necessário deve sempre ser verificado** antes do uso em produção.

## 2.5) spaCy [REVISÃO]

O [spaCy](https://spacy.io/) é o framework de NLP mais usado em produção no mercado.

Ele **não vem com um classificador de sentimento pronto**, mas é excelente para **pré-processamento** (tokenização, lematização, POS tagging) em português — o passo que foi feito manualmente com `frase.lower().split()` na aula anterior.

Aqui, o spaCy é combinado com o léxico do LeIA: o spaCy lematiza (reduz cada palavra à sua forma base) e o LeIA pontua o sentimento.

In [46]:
nlp = spacy.load("pt_core_news_sm")

In [47]:
doc = nlp("Não gostei da cor do produto")
for token in doc:
    print(f"{token.text:12} -> lema: {token.lemma_:12} | classe gramatical: {token.pos_}")

Não          -> lema: não          | classe gramatical: ADV
gostei       -> lema: gostar       | classe gramatical: VERB
da           -> lema: de o         | classe gramatical: ADP
cor          -> lema: cor          | classe gramatical: NOUN
do           -> lema: de o         | classe gramatical: ADP
produto      -> lema: produto      | classe gramatical: NOUN


In [48]:
def classificar_com_spacy_leia(frase):
    doc = nlp(frase)
    # remove stopwords e pontuação, mantém o lema de cada palavra
    frase_lematizada = " ".join(
        token.lemma_ for token in doc if not token.is_stop and not token.is_punct
    )
    return leia.polarity_scores(frase_lematizada)["compound"]

In [49]:
print(f"{'spaCy + LeIA':14} | Frase")
print("-" * 60)
for frase in frases_desafio:
    score = classificar_com_spacy_leia(frase)
    print(f"{score:14.3f} | {frase}")

spaCy + LeIA   | Frase
------------------------------------------------------------
         0.361 | Não gostei da cor do produto.
        -0.250 | Ah, que ótimo, quebrou no primeiro dia!
        -0.402 | Não tive problema nenhum
         0.000 | Mal posso esperar para usar
        -0.103 | Jogo difícil de largar, é muito viciante!!!


## 2.6) Comparando as ferramentas

Tudo é reunido em uma tabela só, lado a lado com o resultado do léxico manual da aula passada.

In [50]:
comparacao_libs = pd.DataFrame({
    "frase": frases_desafio,
    "léxico manual": [classificar_sentimento(f) for f in frases_desafio],
    "NLTK/VADER (compound)": [sia.polarity_scores(f)["compound"] for f in frases_desafio],
    "LeIA (compound)": [leia.polarity_scores(f)["compound"] for f in frases_desafio],
    "spaCy+LeIA (compound)": [classificar_com_spacy_leia(f) for f in frases_desafio],
    "TextBlob (polarity)": [TextBlob(f).sentiment.polarity for f in frases_desafio],
})
comparacao_libs

,frase,léxico manual,NLTK/VADER (compound),LeIA (compound),spaCy+LeIA (compound),TextBlob (polarity)
0,Não gostei da cor do produto.,Positivo,0.0000,0.1531,0.3612,0.0
1,"Ah, que ótimo, quebrou no primeiro dia!",Positivo,-0.3595,-0.1511,-0.2500,0.0
2,Não tive problema nenhum,Negativo,0.0000,-0.5994,-0.4019,0.0
3,Mal posso esperar para usar,Negativo,0.0000,-0.7184,0.0000,0.0
4,"Jogo difícil de largar, é muito viciante!!!",Negativo,0.0000,-0.3129,-0.1027,0.0


### Conclusão do Bloco 2

- **NLTK (VADER)** e **TextBlob** são ótimas ferramentas, extremamente populares — mas **não prontas para português** sem adaptação.
- **LeIA** é a opção gratuita mais direta para um léxico em PT-BR.
- **spaCy** não faz sentimento sozinho, mas é o padrão de mercado para pré-processamento de texto em português, e pode ser combinado com um léxico (como foi feito acima) ou com o classificador **treinado no Bloco 1**.
- Todas elas ainda existe o desafio com **negação** e **sarcasmo** — o mesmo limite observado no léxico manual e no Naive Bayes.

É exatamente aí que os LLMs se destacam, como é mostrado no Bloco 3.

---
# Bloco 3 — IA generativa

Até aqui foram usados: um léxico manual, um modelo de ML treinado do zero, e bibliotecas prontas de mercado. Agora, um **LLM (Large Language Model)** é utilizado — ele entende contexto, negação e até sarcasmo muito melhor, porque foi treinado com uma quantidade massiva de texto e realmente "entende" a língua, em vez de só contar palavras.

## 3.1) Criando uma API key na Groq Console

1. Acesse **https://console.groq.com/keys** e faça login (dá para usar conta Google, GitHub, etc.).
2. Clique em **"Create API Key"**.
3. Dê um nome para a chave (ex: `aula-sentimento`) e clique em **"Submit"**.
4. Copie a chave gerada (uma string começando com `gsk_...`). **Trate essa chave como uma senha** — quem tiver acesso a ela pode consumir sua cota gratuita.
5. O **free tier** da Groq não exige cartão de crédito e permite um número generoso de requisições por dia — mais do que suficiente para esta aula (consulte os limites atuais em https://console.groq.com/docs/rate-limits).

### Guardando a chave com segurança no Google Colab

Se você está rodando este notebook no **Google Colab**, não cole a chave direto no código. Use o recurso de **Secrets** do Colab:

1. Clique no ícone de **chave (🔑) "Secrets"** no menu lateral esquerdo do Colab.
2. Clique em **"Add new secret"**.
3. Em **Name**, coloque `GROQ_API_KEY`. Em **Value**, cole a chave copiada da Groq Console.
4. Ative o toggle **"Notebook access"** para este notebook.
5. Recupere a chave no código com `google.colab.userdata`, como abaixo — ela nunca fica visível ou salva no arquivo `.ipynb`.

In [57]:
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODELO_GROQ = "llama-3.1-70b-versatile"

## 3.2) Prompt simples (zero-shot)

A forma mais direta: descrever a tarefa e pedir a classificação, sem exemplos.

In [58]:
def classificar_com_llm(frase):
    prompt = f"""Classifique o sentimento da frase abaixo como apenas uma destas opções:
Positivo, Negativo ou Neutro.

Responda apenas com a palavra da classificação, sem explicações.

Frase: "{frase}"
"""
    resposta = client.chat.completions.create(
        model=MODELO_GROQ,
        messages=[{"role": "user", "content": prompt}],
    )
    return resposta.choices[0].message.content.strip()

In [59]:
classificar_com_llm("Não gostei da cor do produto.")

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [61]:
classificar_com_llm("O produto é ok")

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

## 3.3) Prompt com exemplos (few-shot)

Alguns exemplos de frase → classificação são fornecidos antes de pedir a nova classificação. Isso ajuda o modelo a calibrar o formato e o critério de resposta, especialmente em casos ambíguos.

In [60]:
# def classificar_few_shot(frase):
#     prompt = f"""Classifique o sentimento de frases sobre produtos como Positivo, Negativo ou Neutro.

# Exemplos:
# Frase: "Adorei o produto, chegou rápido!"
# Classificação: Positivo

# Frase: "Não gostei da cor do produto."
# Classificação: Negativo

# Frase: "Não tive problema nenhum."
# Classificação: Positivo

# Frase: "O relatório foi entregue."
# Classificação: Neutro

# Agora classifique a frase abaixo. Responda apenas com a palavra da classificação.

# Frase: "{frase}"
# Classificação:"""
#     resposta = client.models.generate_content(model=MODELO_GEMINI, contents=prompt)
#     return resposta.text.strip()

In [62]:
classificar_com_llm("Seria bom se não travasse")

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

## 3.4) Para se aprofundar: LLMs para análise de sentimento

Alguns materiais para continuar estudando o uso de LLMs em tarefas de classificação de texto:



- 📘 [Groq — Quickstart](https://console.groq.com/docs/quickstart) (inglês) — guia oficial de início rápido com a API da Groq.
- 📘 [Groq — Modelos disponíveis](https://console.groq.com/docs/models) (inglês) — lista dos modelos (Llama e outros) disponíveis para uso na API.
- 📘 [Prompt Engineering Guide — Few-shot Prompting](https://www.promptingguide.ai/techniques/fewshot) (inglês) — aprofunda a técnica de few-shot usada acima, com mais exemplos e variações.
- 📘 [Google Cloud — Analisando sentimento de texto](https://cloud.google.com/natural-language/docs/analyzing-sentiment) (inglês, com opção de tradução no próprio site) — mostra a abordagem de uma API de NLP tradicional (não generativa) para sentimento, um bom contraponto ao que foi feito com o LLM da Groq.
- 📘 [Gemini API — Text generation](https://ai.google.dev/gemini-api/docs/text-generation) (inglês) — guia oficial de geração de texto e prompting com o Gemini.
- 📘 [Gemini API — Text generation (pt-BR)](https://ai.google.dev/gemini-api/docs/text-generation?hl=pt-br) (português) — a mesma documentação oficial, traduzida.
- 📘 [Prompt Engineering Guide — Few-shot Prompting](https://www.promptingguide.ai/techniques/fewshot) (inglês) — aprofunda a técnica de few-shot usada acima, com mais exemplos e variações.
- 📘 [Google Cloud — Analisando sentimento de texto](https://cloud.google.com/natural-language/docs/analyzing-sentiment) (inglês, com opção de tradução no próprio site) — mostra a abordagem de uma API de NLP tradicional (não generativa) para sentimento, um bom contraponto ao que foi feito com o Gemini.

### Discussão final

- O **léxico manual** é rápido de entender e não precisa de dados de treino, mas é ingênuo: não entende negação nem contexto.
- O **ML treinado** aprende padrões automaticamente a partir de exemplos, mas ainda depende de **quantidade e qualidade dos dados de treino**, e o Bag of Words continua sem entender negação.
- O **LLM** entende contexto, negação e até sarcasmo, além de explicar o motivo da classificação, se pedirmos — mas tem **custo por chamada** (mesmo que pequeno no free tier), **latência maior** (chamada de rede) e depende de um serviço externo.

# Importante: Não existe a melhor ferramenta para todos os cenários.
>
> A escolha da tecnologia deve considerar fatores como:
>
> - 💰 **Custo**
> - 📊 **Volume de dados**
> - 🔍 **Necessidade de explicabilidade**
> - ⚡ **Latência aceitável**
>